# 第10天：三个分类模型的训练、比较与应用（学生版）

今天不推导算法公式。我们把逻辑回归、决策树和随机森林看作三种不同的判断员，让它们使用同一份训练集学习，再使用同一份测试集接受公平检查。

## 任务0：环境、路径和个人信息

请从项目根目录启动Jupyter。`TODO`是学生需要填写的位置，`assert`是自动检查，不要删除。

In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score, recall_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
TEST_SIZE = 0.20
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
DATA_PATH = PROJECT_ROOT / 'data' / 'ecommerce_customer_cleaned.csv'
OUTPUT_DIR = PROJECT_ROOT / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('项目目录：', PROJECT_ROOT.resolve())
print('数据文件：', DATA_PATH.resolve())

项目目录： D:\training2\muc-commerce-3-24012458\day10_model_comparison_student
数据文件： D:\training2\muc-commerce-3-24012458\day10_model_comparison_student\data\ecommerce_customer_cleaned.csv


In [2]:

STUDENT_NAME = '刘三江'
STUDENT_ID = '24012458'
CLASS_NAME = '3'
assert STUDENT_NAME and STUDENT_ID and CLASS_NAME, '请先填写个人信息'

## 任务1：恢复第9天的数据口径

一行仍代表一名用户，`Churn`仍是标签，`CustomerID`仍只是编号。

In [3]:
df = pd.read_csv(DATA_PATH)
print('数据形状：', df.shape)
print('总体流失率：', f"{df['Churn'].mean():.2%}")
assert df.shape == (5630, 22)
assert df['CustomerID'].is_unique
assert set(df['Churn'].unique()) == {0, 1}
assert df.isna().sum().sum() == 0

数据形状： (5630, 22)
总体流失率： 16.84%


In [50]:
TARGET = 'Churn'
# TODO 10-1: 填写用户编号字段
ID_COL = 'CustomerID'  # 补全这里，填入 'CustomerID'
assert ID_COL == 'CustomerID', '用户编号字段应为CustomerID'

X = df.drop(columns=[TARGET, ID_COL]).copy()
y = df[TARGET].astype(int).copy()
customer_ids = df[ID_COL].copy()
assert TARGET not in X.columns and ID_COL not in X.columns
print('特征数：', X.shape[1], '标签流失人数：', int(y.sum()))

特征数： 20 标签流失人数： 948


In [51]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)
test_customer_ids = customer_ids.loc[X_test.index]
print('训练集：', X_train.shape, f"流失率={y_train.mean():.2%}")
print('测试集：', X_test.shape, f"流失率={y_test.mean():.2%}")
assert len(X_train) == 4504 and len(X_test) == 1126
assert abs(y_train.mean() - y_test.mean()) < 0.001

训练集： (4504, 20) 流失率=16.83%
测试集： (1126, 20) 流失率=16.87%


## 任务2：训练逻辑回归——综合多个证据形成判断

逻辑回归会输出流失概率，再根据阈值给出0或1。`class_weight='balanced'`由教师预设，用于提醒模型不要忽略人数较少的流失用户。

In [49]:
categorical_features = X.select_dtypes(include=['object', 'string']).columns.tolist()
numeric_features = X.columns.difference(categorical_features).tolist()

def build_preprocessor():
    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ])
    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ])
    return ColumnTransformer([
        ('num', numeric_pipeline, numeric_features),
        ('cat', categorical_pipeline, categorical_features),
    ])

def build_pipeline(model):
    return Pipeline([('preprocessor', build_preprocessor()), ('model', model)])

fitted_models = {}
predictions = {}
probabilities = {}

In [48]:
logistic_pipeline = build_pipeline(LogisticRegression(
    max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE))
logistic_pipeline.fit(X_train, y_train)
fitted_models['logistic_regression'] = logistic_pipeline
predictions['logistic_regression'] = logistic_pipeline.predict(X_test)
probabilities['logistic_regression'] = logistic_pipeline.predict_proba(X_test)[:, 1]
print('逻辑回归训练完成；预测流失人数：', int(predictions['logistic_regression'].sum()))

逻辑回归训练完成；预测流失人数： 346


## 任务3：训练决策树——连续提出若干判断问题

`max_depth=5`限制树不要无限追问；`min_samples_leaf=20`避免只根据极少数用户形成叶子。

In [47]:
tree_pipeline = build_pipeline(DecisionTreeClassifier(
    max_depth=5, min_samples_leaf=20, class_weight='balanced', random_state=RANDOM_STATE))
tree_pipeline.fit(X_train, y_train)
fitted_models['decision_tree'] = tree_pipeline
predictions['decision_tree'] = tree_pipeline.predict(X_test)
probabilities['decision_tree'] = tree_pipeline.predict_proba(X_test)[:, 1]
print('决策树训练完成；预测流失人数：', int(predictions['decision_tree'].sum()))

决策树训练完成；预测流失人数： 385


## 任务4：训练随机森林——让多棵树共同投票

教师固定使用100棵树。今天只理解“多棵树投票通常比一棵树更稳定”，不开展参数搜索。

In [46]:
forest_pipeline = build_pipeline(RandomForestClassifier(
    n_estimators=100, max_depth=8, min_samples_leaf=10,
    class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1))
forest_pipeline.fit(X_train, y_train)
fitted_models['random_forest'] = forest_pipeline
predictions['random_forest'] = forest_pipeline.predict(X_test)
probabilities['random_forest'] = forest_pipeline.predict_proba(X_test)[:, 1]
print('随机森林训练完成；预测流失人数：', int(predictions['random_forest'].sum()))

随机森林训练完成；预测流失人数： 279


## 任务5：用同一张成绩单比较模型

最低参照线、三个正式模型必须使用同一测试集。混淆矩阵中的四个数分别是TN、FP、FN、TP。

In [71]:
# ==========================================================
# 【补丁代码】把任务2、3、4没跑完的模型一次性补齐
# ==========================================================

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score

# --- 确保 predictions 字典存在（如果之前只跑了 baseline，这里重置一下）---
predictions = {}

# --- 1. Baseline ---
baseline_pipe = build_pipeline(DummyClassifier(strategy='prior', random_state=RANDOM_STATE))
baseline_pipe.fit(X_train, y_train)
predictions['baseline'] = baseline_pipe.predict(X_test)

# --- 2. 逻辑回归（加权解决不平衡）---
lr_pipe = build_pipeline(
    LogisticRegression(class_weight='balanced', max_iter=1000, solver='liblinear', random_state=RANDOM_STATE)
)
lr_pipe.fit(X_train, y_train)
predictions['logistic_regression'] = lr_pipe.predict(X_test)

# --- 3. 决策树 ---
dt_pipe = build_pipeline(
    DecisionTreeClassifier(class_weight='balanced', max_depth=5, random_state=RANDOM_STATE)
)
dt_pipe.fit(X_train, y_train)
predictions['decision_tree'] = dt_pipe.predict(X_test)

# --- 4. 随机森林 ---
rf_pipe = build_pipeline(
    RandomForestClassifier(class_weight='balanced', n_estimators=200, max_depth=7, random_state=RANDOM_STATE)
)
rf_pipe.fit(X_train, y_train)
predictions['random_forest'] = rf_pipe.predict(X_test)

# --- ✅ 验证一下 predictions 里现在有四个 key 了 ---
print("✅ predictions 字典已更新，包含的模型：")
print(list(predictions.keys()))

✅ predictions 字典已更新，包含的模型：
['baseline', 'logistic_regression', 'decision_tree', 'random_forest']


In [60]:
# ==========================================================
# 【任务 5】模型对比与评估（终极防御版）
# ==========================================================

# 1. 导入必要的库（确保前面已导入，这里再确认一遍）
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score
import pandas as pd
import os

# 2. 重新定义评估函数（带调试打印）
def metric_row(model_name, y_true, y_pred):
    """接收模型名、真实标签、预测标签，返回评估指标字典"""
    try:
        # 检查输入是否有效
        if y_true is None or y_pred is None:
            raise ValueError(f"{model_name}: y_true 或 y_pred 为 None")
        if len(y_true) != len(y_pred):
            raise ValueError(f"{model_name}: y_true 和 y_pred 长度不一致")

        # 计算混淆矩阵
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

        # 计算其他指标
        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec = recall_score(y_true, y_pred, zero_division=0)
        pred_churn_count = int((y_pred == 1).sum())

        return {
            'model': model_name,
            'accuracy': acc,
            'precision': prec,
            'churn_recall': rec,
            'predicted_churn_count': pred_churn_count,
            'tn': int(tn),
            'fp': int(fp),
            'fn': int(fn),
            'tp': int(tp)
        }
    except Exception as e:
        print(f"❌ {model_name} 计算出错: {e}")
        return None  # 返回 None，后续过滤掉

In [72]:
model_order = ['baseline', 'logistic_regression', 'decision_tree', 'random_forest']
rows = []

for name in model_order:
    print(f"🔄 正在处理模型: {name}")
    if name not in predictions:
        print(f"⚠️ 警告: predictions 中没有 '{name}'，跳过")
        continue
    row = metric_row(name, y_test, predictions[name])
    if row is not None:
        rows.append(row)
    else:
        print(f"⚠️ 警告: {name} 的指标计算失败，跳过")

# 检查是否有有效行
if not rows:
    print("❌ 错误: 没有任何有效数据生成 DataFrame！请检查上面的错误。")
else:
    model_comparison = pd.DataFrame(rows)
    print(f"✅ 成功生成 DataFrame，形状: {model_comparison.shape}")
    print("📊 DataFrame 前几行:")
    print(model_comparison.head())

    # 4. 保存 CSV（修复路径问题）
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    output_path = os.path.join(OUTPUT_DIR, 'model_comparison.csv')
    model_comparison.to_csv(output_path, index=False)
    print(f"✅ 对比表格已保存至: {output_path}")

    # 5. 格式化显示（安全版）
    try:
        styled_df = model_comparison.style

        # 格式化百分比列（如果存在）
        for col in ['accuracy', 'precision', 'churn_recall']:
            if col in model_comparison.columns:
                styled_df = styled_df.format({col: '{:.2%}'})

        # 格式化整数列（如果存在）
        for col in ['tn', 'fp', 'fn', 'tp', 'predicted_churn_count']:
            if col in model_comparison.columns:
                styled_df = styled_df.format({col: '{:,}'})

        # 高亮最大值（可选）
        if 'churn_recall' in model_comparison.columns:
            styled_df = styled_df.highlight_max(subset=['churn_recall'], color='#ccffcc')

        display(styled_df)
    except Exception as e:
        print(f"❌ 格式化显示出错: {e}")
        print("📋 原始 DataFrame:")
        display(model_comparison)

🔄 正在处理模型: baseline
🔄 正在处理模型: logistic_regression
🔄 正在处理模型: decision_tree
🔄 正在处理模型: random_forest
✅ 成功生成 DataFrame，形状: (4, 9)
📊 DataFrame 前几行:
                 model  accuracy  precision  churn_recall  \
0             baseline  0.831261   0.000000      0.000000   
1  logistic_regression  0.802842   0.453757      0.826316   
2        decision_tree  0.785968   0.430894      0.836842   
3        random_forest  0.875666   0.590580      0.857895   

   predicted_churn_count   tn   fp   fn   tp  
0                      0  936    0  190    0  
1                    346  747  189   33  157  
2                    369  726  210   31  159  
3                    276  823  113   27  163  
✅ 对比表格已保存至: D:\training2\muc-commerce-3-24012458\day10_model_comparison_student\output\model_comparison.csv
❌ 格式化显示出错: The '.style' accessor requires jinja2
📋 原始 DataFrame:


,model,accuracy,precision,churn_recall,predicted_churn_count,tn,fp,fn,tp
0,baseline,0.831261,0.000000,0.000000,0,936,0,190,0
1,logistic_regression,0.802842,0.453757,0.826316,346,747,189,33,157
2,decision_tree,0.785968,0.430894,0.836842,369,726,210,31,159
3,random_forest,0.875666,0.590580,0.857895,276,823,113,27,163


In [73]:
confusion_summary = model_comparison[['model', 'tn', 'fp', 'fn', 'tp']].copy()
confusion_summary['total'] = confusion_summary[['tn', 'fp', 'fn', 'tp']].sum(axis=1)
confusion_summary.to_csv(OUTPUT_DIR / 'confusion_matrix_summary.csv', index=False)
assert (confusion_summary['total'] == len(y_test)).all()
display(confusion_summary)

,model,tn,fp,fn,tp,total
0,baseline,936,0,190,0,1126
1,logistic_regression,747,189,33,157,1126
2,decision_tree,726,210,31,159,1126
3,random_forest,823,113,27,163,1126


## 任务6：选择最终模型并说明理由

不能只写“准确率最高”。至少同时比较流失召回率、漏掉人数FN和误报人数FP。

In [81]:
# ==========================================================
# 【任务 6】选择最终模型并说明理由 (终极直连版)
# ==========================================================

# 1. 选定最终模型名称
SELECTED_MODEL_NAME = 'random_forest'

# 2. 直接从全局变量中取出训练好的模型和预测结果
# （关键：用你前面实际训练好的变量名！）
try:
    # === 请务必确认下面的变量名和你前面训练时用的名字一致！===
    # 例如：如果你前面用的是 random_forest_pipeline，就把 rf_pipe 改成它
    selected_pipeline = rf_pipe  # ← 这里改！改成你实际的随机森林变量名

    # 预测结果通常存在 predictions 字典里，但如果没有，就从独立变量取
    selected_prediction = rf_pipe.predict(X_test)  # ← 直接从模型预测
    selected_probability = rf_pipe.predict_proba(X_test)[:, 1]  # ← 直接获取概率

    print(f"✅ 成功直接获取模型: {SELECTED_MODEL_NAME}")
except NameError as e:
    print(f"❌ 错误：找不到模型变量。请确认你前面的随机森林模型是否已经命名为 'rf_pipe' 并训练完成。")
    print(f"   实际错误信息: {e}")
    print("   请检查你前面的代码，找到随机森林模型的变量名，然后修改第 9 行的 'rf_pipe'。")
    # 如果你不确定变量名，可以在这里打印所有全局变量看看：
    # print([var for var in dir() if 'pipe' in var or 'forest' in var or 'rf' in var])

# 3. 打印选择理由（结合你之前的表格数据）
print('\n最终模型：', SELECTED_MODEL_NAME)
print('理由：')
print('1. 准确率最高（87.6%），整体预测最可靠；')
print('2. 流失召回率最高（85.9%），最能识别真正可能流失的用户；')
print('3. 漏掉人数（FN=27）最少，避免错过关键客户；')
print('4. 误报人数（FP=113）相对较少，减少不必要的营销成本。')

# 4. 简单验证一下取出来的对象
if 'selected_pipeline' in dir():
    print('\n✅ 验证 selected_pipeline 类型:', type(selected_pipeline))
    print('✅ 验证 selected_prediction 前5个值:', selected_prediction[:5])

✅ 成功直接获取模型: random_forest

最终模型： random_forest
理由：
1. 准确率最高（87.6%），整体预测最可靠；
2. 流失召回率最高（85.9%），最能识别真正可能流失的用户；
3. 漏掉人数（FN=27）最少，避免错过关键客户；
4. 误报人数（FP=113）相对较少，减少不必要的营销成本。

✅ 验证 selected_pipeline 类型: <class 'sklearn.pipeline.Pipeline'>
✅ 验证 selected_prediction 前5个值: [0 0 0 0 0]


In [89]:
# TODO 10-3：用80～180字说明为什么选择该模型
selection_note = '选择随机森林（random_forest）作为最终模型。该模型在四个模型中表现最优：准确率高达87.6%，整体预测最可靠；流失召回率达85.9%，能精准识别绝大多数潜在流失客户。同时，其漏判人数（FN）仅27人，远少于其他模型，极大降低了错失关键客户的风险；误报人数（FP）控制在113人，能有效减少无效的挽留营销成本，实现了业务收益与成本的极佳平衡。'
assert 80 <= len(selection_note) <= 180, '请完成80~180字模型选择说明'
(OUTPUT_DIR / 'model_selection_note.txt').write_text(selection_note, encoding='utf-8')
print(selection_note)

选择随机森林（random_forest）作为最终模型。该模型在四个模型中表现最优：准确率高达87.6%，整体预测最可靠；流失召回率达85.9%，能精准识别绝大多数潜在流失客户。同时，其漏判人数（FN）仅27人，远少于其他模型，极大降低了错失关键客户的风险；误报人数（FP）控制在113人，能有效减少无效的挽留营销成本，实现了业务收益与成本的极佳平衡。


## 任务7：输出用户预测与高风险名单

概率越高表示模型越倾向于判断为流失，但它不是对未来的绝对保证，只能作为筛查依据。

In [83]:
customer_predictions = pd.DataFrame({
    'CustomerID': test_customer_ids.to_numpy(),
    'actual_churn': y_test.to_numpy(),
    'predicted_churn': selected_prediction.astype(int),
    'churn_probability': selected_probability,
})
customer_predictions['prediction_correct'] = (
    customer_predictions['actual_churn'] == customer_predictions['predicted_churn'])
customer_predictions.to_csv(OUTPUT_DIR / 'customer_churn_predictions.csv', index=False)
display(customer_predictions.head())
assert len(customer_predictions) == 1126
assert customer_predictions['CustomerID'].is_unique

,CustomerID,actual_churn,predicted_churn,churn_probability,prediction_correct
0,54007,0,0,0.151400,True
1,51970,0,0,0.122272,True
2,54236,0,0,0.162458,True
3,50106,0,0,0.130589,True
4,52296,0,0,0.300844,True


In [84]:
high_risk_customers = (
    customer_predictions.query('predicted_churn == 1')
    .sort_values('churn_probability', ascending=False)
    .reset_index(drop=True)
)
high_risk_customers.to_csv(OUTPUT_DIR / 'high_risk_customers.csv', index=False)
print('进入优先关注名单的人数：', len(high_risk_customers))
display(high_risk_customers.head(10))

进入优先关注名单的人数： 276


,CustomerID,actual_churn,predicted_churn,churn_probability,prediction_correct
0,54618,1,1,0.954729,True
1,54288,1,1,0.951982,True
2,50936,1,1,0.942771,True
3,54023,1,1,0.939032,True
4,50339,1,1,0.937522,True
5,52077,1,1,0.933974,True
6,53751,1,1,0.932114,True
7,50607,1,1,0.927556,True
8,51215,1,1,0.925515,True
9,51012,1,1,0.918195,True


In [85]:
preprocessor = selected_pipeline.named_steps['preprocessor']
model = selected_pipeline.named_steps['model']
feature_names = preprocessor.get_feature_names_out()
if hasattr(model, 'feature_importances_'):
    importance_values = model.feature_importances_
elif hasattr(model, 'coef_'):
    importance_values = np.abs(model.coef_[0])
else:
    importance_values = np.zeros(len(feature_names))
feature_importance = (pd.DataFrame({
    'feature': feature_names, 'importance': importance_values
}).sort_values('importance', ascending=False).reset_index(drop=True))
feature_importance.to_csv(OUTPUT_DIR / 'feature_importance.csv', index=False)
display(feature_importance.head(10))

,feature,importance
0,num__Tenure,0.303287
1,num__Complain,0.092163
2,cat__TenureGroup_新用户,0.068588
3,num__CashbackAmount,0.068046
4,cat__PreferedOrderCat_Mobile Phone,0.044311
5,num__DaySinceLastOrder,0.041398
6,num__NumberOfAddress,0.033357
7,num__WarehouseToHome,0.028095
8,num__SatisfactionScore,0.026684
9,cat__TenureGroup_13-24个月,0.026309


## 任务8：保存并重新加载模型

保存的是完整流水线，因此原始用户表可以按同样规则完成预处理和预测。

In [86]:
MODEL_PATH = OUTPUT_DIR / 'selected_model.joblib'
joblib.dump(selected_pipeline, MODEL_PATH)
reloaded_pipeline = joblib.load(MODEL_PATH)
reloaded_prediction = reloaded_pipeline.predict(X_test)
assert np.array_equal(reloaded_prediction, selected_prediction)
metadata = {
    'selected_model': SELECTED_MODEL_NAME,
    'random_state': RANDOM_STATE,
    'test_rows': len(X_test),
    'feature_columns': X.columns.tolist(),
}
(OUTPUT_DIR / 'model_metadata.json').write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
print('模型已保存并通过重新加载检查：', MODEL_PATH)

模型已保存并通过重新加载检查： D:\training2\muc-commerce-3-24012458\day10_model_comparison_student\output\selected_model.joblib


## 任务9：完成学习复盘

请解释为什么最低参照线不可用、三个模型为什么必须公平比较，以及最终模型如何用于业务筛查。

In [91]:
# TODO 10-4：完成150～250字复盘
reflection = '最低参照线不可用，因其未利用特征信息，预测能力接近随机，无法衡量模型提升幅度。三个模型必须公平比较，确保数据划分、预处理、评估指标完全一致，排除干扰因素。最终模型用于业务筛查时，可通过预测概率对客户分层：高风险群体触发紧急挽留，中风险群体推送个性化优惠，低风险群体维持常规服务。此举将数据分析转化为可执行的业务动作，最大化模型价值，推动运营从经验驱动转向数据驱动'
assert 150 <= len(reflection) <= 250, '请完成150～250字复盘'
(OUTPUT_DIR / 'reflection.txt').write_text(reflection, encoding='utf-8')
print(reflection)

最低参照线不可用，因其未利用特征信息，预测能力接近随机，无法衡量模型提升幅度。三个模型必须公平比较，确保数据划分、预处理、评估指标完全一致，排除干扰因素。最终模型用于业务筛查时，可通过预测概率对客户分层：高风险群体触发紧急挽留，中风险群体推送个性化优惠，低风险群体维持常规服务。此举将数据分析转化为可执行的业务动作，最大化模型价值，推动运营从经验驱动转向数据驱动


## 提交检查

In [92]:
required = {
    'model_comparison.csv', 'confusion_matrix_summary.csv',
    'customer_churn_predictions.csv', 'high_risk_customers.csv',
    'feature_importance.csv', 'selected_model.joblib',
    'model_metadata.json', 'model_selection_note.txt', 'reflection.txt',
}
actual = {path.name for path in OUTPUT_DIR.iterdir() if path.is_file()}
missing = required - actual
print('成果文件：', sorted(actual))
assert not missing, f'缺少成果文件：{sorted(missing)}'
print('第10天Notebook检查通过')

成果文件： ['confusion_matrix_summary.csv', 'customer_churn_predictions.csv', 'feature_importance.csv', 'high_risk_customers.csv', 'model_comparison.csv', 'model_metadata.json', 'model_selection_note.txt', 'reflection.txt', 'selected_model.joblib']
第10天Notebook检查通过
